# StorageGuard: continue the study

Use this notebook for every session after the first. The first session installed everything and Qwen2.5-7B passed its smoke test. Mistral-7B-Instruct-v0.3 was dropped: it cannot drive the tool interface (it writes command lines instead of tool calls), so Mistral's current small model, **Ministral 3 8B** (2512, Apache-2.0), takes its place. Llama 3.1 waits for Meta's approval.

| Part | Cells | What | Time on an A100 (estimate) |
|---|---|---|---|
| A | 1-3 | GPU check, Drive, install (a new runtime starts empty) | ~10 min |
| A | 4 | smoke test Ministral 3 8B (first download ~18 GB) | ~5-10 min |
| A | 5 | full run: Qwen 7B (dev, main, ablations) and Ministral 3 8B (main), then the report | ~1.5-2.5 h |
| A | 6-7 | look at the report, download the interim results | 1 min |
| B | 8-10 | **scale study** (recommended): Qwen2.5 14B, 32B and 72B, plus a 4-bit 7B control (80 GB A100) | ~1.5-2 h (1 run each) |
| C | 11-14 | **Llama 3.1, once Meta approves**: check access, smoke test, main runs, final report and download | ~45 min |

**Before you start**

1. If another notebook is still connected, stop its runtime there: **Runtime -> Disconnect and delete runtime**. Otherwise two A100s are billed at once.
2. Upload the new `storageguard_colab.zip` into `MyDrive/storageguard/` and choose **Replace existing file** if Drive asks (cell 2 uses the newest zip either way).
3. **Runtime -> Change runtime type -> A100 GPU**.

Parts B and C are independent: run them in any order, in this session (after cell 5) or in a later one (then run cells 1, 2, 3 first). Nothing finished is ever run twice: results are mirrored to `MyDrive/storageguard/state/` and restored at the start of every run.

## 1. Check the GPU

In [ ]:
import subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
print(gpu or "NO GPU")
assert gpu, "No GPU: Runtime -> Change runtime type -> A100 GPU"
if "A100" not in gpu and "H100" not in gpu:
    print("WARNING: not an A100/H100. 7-8B models still fit on a 24 GB GPU (L4), but expect it to be slower.")

## 2. Settings, Google Drive and the project

In [ ]:
# ---- settings -------------------------------------------------------------
BACKEND   = "vllm"
MODELS    = "qwen2.5-7b,ministral-3-8b"             # Part A: the main models that need no approval
SCALE     = "qwen2.5-7b-awq,qwen2.5-14b,qwen2.5-32b-int8,qwen2.5-72b-awq"  # Part B (80 GB A100)
# on a 40 GB A100: SCALE = "qwen2.5-7b-awq,qwen2.5-14b-int8,qwen2.5-32b-awq"
LATER     = "llama3.1-8b"                           # Part C: once Meta approves the licence
MAIN_RUNS = 3                                       # runs per (model, config) for the main models
SCALE_RUNS = 1                                      # runs per config in Part B; 3 matches Part A (~3.5-4 h instead of ~1.5-2 h)
DRIVE_DIR = "/content/drive/MyDrive/storageguard"   # where storageguard_colab.zip is
# ---------------------------------------------------------------------------

import os, sys, time, zipfile, pathlib, subprocess
from google.colab import drive, userdata

drive.mount("/content/drive")
bundles = sorted(pathlib.Path(DRIVE_DIR).glob("storageguard_colab*.zip"), key=os.path.getmtime)
assert bundles, f"Upload storageguard_colab.zip to {DRIVE_DIR} first"
bundle = bundles[-1]                          # the newest upload, also if Drive kept an older copy
zipfile.ZipFile(bundle).extractall("/content")  # always the newest code; results are never in the zip
print("code from", bundle.name, "uploaded", time.strftime("%Y-%m-%d %H:%M", time.localtime(bundle.stat().st_mtime)))
os.chdir("/content/storageguard")
os.environ["SG_DRIVE_STATE"] = f"{DRIVE_DIR}/state"
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN loaded")
except Exception as exc:  # secret missing or notebook access not granted
    print(f"No HF_TOKEN ({type(exc).__name__}). Qwen and Mistral will work; Llama downloads will fail.")

def run(*args):
    """Run the driver, stream its output here, and stop the notebook if it fails."""
    p = subprocess.Popen([sys.executable, "-u", "colab/sg_colab.py", *map(str, args)],
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"step failed with exit code {p.returncode}; read the messages above")

def show_report():
    from IPython.display import Markdown, Image, display
    display(Markdown(pathlib.Path("results/REPORT.md").read_text()))
    for png in sorted(pathlib.Path("paper/figures").glob("*.png")):
        print(png.name)
        display(Image(filename=str(png), width=900))

def download_results():
    run("pack")   # zips results/, paper/, data/bench and configs into DRIVE_DIR
    from google.colab import files
    latest = max(pathlib.Path(DRIVE_DIR).glob("storageguard_results_*.zip"), key=os.path.getmtime)
    print("saved to Drive:", latest)
    files.download(str(latest))

print("project ready at", os.getcwd())

## 3. Install (about 5-10 minutes)

Creates two isolated environments: one for the experiment harness (Python 3.11) and one for vLLM (Python 3.12), then runs the fast unit tests.

In [ ]:
run("setup", "--backend", BACKEND)

## 4. Smoke test Ministral 3 8B

Qwen passed in the first session, so only Ministral is tested here (12 short episodes). It must end with `smoke test PASSED`. If it fails, the output shows what the model wrote instead of tool calls; continue with Qwen alone (`MODELS = "qwen2.5-7b"` in cell 2, run cell 2 again, then cell 5).

In [ ]:
run("smoke", "--backend", BACKEND, "--models", "ministral-3-8b")

## 5. Full run: Qwen and Ministral

Qwen2.5-7B: dev split (9 configs), main matrix (5 configs x 3 runs), ablations, threshold sweep and approver sensitivity. Ministral 3 8B: main matrix. Then the report.

Keep the tab open. If the session disconnects: run cells 1, 2, 3, then this cell again; finished episodes are skipped. Progress lines look like `600/1560  0.9s/episode  eta 14 min`.

In [ ]:
run("all", "--backend", BACKEND, "--models", MODELS, "--main-runs", MAIN_RUNS)

## 6. Report (so far)

In [ ]:
show_report()

## 7. Download the interim results

A copy is also saved in `MyDrive/storageguard/`.

In [ ]:
download_results()

---
# Part B: scale study (recommended)

Does a larger model still need the guard? Same family (Qwen2.5), same tool format, four sizes on an 80 GB A100: 7B (Part A, full precision), **14B** (full precision, 29.5 GB), **32B** (official GPTQ 8-bit, 35.1 GB; near-lossless) and **72B** (official AWQ 4-bit, 41.6 GB). Larger precisions do not fit with room for the KV cache. The **7B in AWQ 4-bit** is a control: comparing it with the full-precision 7B shows how much 4-bit weights alone change the results.

Main matrix only (5 configs), `SCALE_RUNS` runs each. About 120 GB of downloads: when the disk runs short, the driver deletes the cached weights of models that are already finished (they would simply be downloaded again if needed). A model too large for the GPU is rejected in seconds with the 40 GB alternative.

## 8. Check the larger models can run here (seconds)

GPU memory and download access for each model.

In [ ]:
run("access", "--models", SCALE)

## 9. Smoke test and main runs, model by model

Each model is downloaded and loaded once: smoke test first, its runs only if the smoke test passes. Roughly: 7B-AWQ 5 min, 14B 10 min, 32B 20 min, 72B 40-60 min with one run each. If a model failed cell 8, leave it out, e.g. `run("all", "--backend", BACKEND, "--models", "qwen2.5-14b,qwen2.5-32b-int8", "--main-runs", SCALE_RUNS)`.

In [ ]:
run("all", "--backend", BACKEND, "--models", SCALE, "--main-runs", SCALE_RUNS, "--smoke-first")

## 10. Report and download

In [ ]:
show_report()
download_results()

---
# Part C: Llama 3.1 (after Meta approves)

You will get an e-mail from Hugging Face, and the model page (<https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct>) will say *You have been granted access*. The `HF_TOKEN` secret must be set for this notebook (key icon, **Notebook access** on); if you added it after running cell 2, run cell 2 again.

In a new session: run cells 1, 2 and 3 first, then continue here.

## 11. Has Meta approved it yet?

Takes a second. Prints `OK, can be downloaded`, or what is still missing.

In [ ]:
run("access", "--models", LATER)

## 12. Smoke test Llama

In [ ]:
run("smoke", "--backend", BACKEND, "--models", LATER)

## 13. Main runs for Llama (about 30-40 min)

Only Llama's episodes run. Earlier results are restored from Drive and kept, and the report is rebuilt over all models.

In [ ]:
run("all", "--backend", BACKEND, "--models", LATER, "--main-runs", MAIN_RUNS)

## 14. Final report and download

In [ ]:
show_report()
download_results()

## Troubleshooting

| Symptom | Fix |
|---|---|
| Cell 2: `Upload storageguard_colab.zip to ... first` | The zip is not in `MyDrive/storageguard/`, or `DRIVE_DIR` points elsewhere. |
| Session disconnected | Run cells 1, 2, 3, then the cell that was running. Nothing finished is lost. |
| A smoke test fails | The output lists the problems and shows replies the model wrote without a tool call. Leave that model out and report it as a limitation. |
| Cell 11: `NO ACCESS` | Not approved yet (the model page says "awaiting review"), or the token is fine-grained without "Read access to contents of all public gated repos": use a classic *Read* token, update the secret, run cell 2 again. |
| vLLM install fails or the server exits at start | Read the log tail printed above. Setup installs the pinned vLLM 0.33.0 with matching packages; keep it for the whole study. Last resort: `BACKEND = "ollama"` (results stored as `<model>-q4`, never mixed with vLLM ones). |
| Mistral-format model: `400 ... missing ... 'force_tool_names'` | mistral_common too new for vLLM 0.33.0: run cell 3 again with the current zip, or `!uv pip install --python /content/venv-vllm/bin/python "mistral-common[image]==1.12.0"`. |
| CUDA out of memory | Add `"--concurrency", 8` to the `run(...)` call. |
| Want to check progress | `!find results/runs -name '*.json' | wc -l` in a new cell. |